In [19]:
print("""
@File         : group_by_basics.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 15:34:57
@Email        : cuixuanstephen@gmail.com
@Description  : Group by basics
""")


@File         : group_by_basics.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 15:34:57
@Email        : cuixuanstephen@gmail.com
@Description  : Group by basics



In [20]:
import pandas as pd

In [21]:
df = pd.DataFrame([
    ["group_a", 0],
    ["group_a", 2],
    ["group_b", 1],
    ["group_b", 3],
    ["group_b", 5],
], columns=["group", "value"])
df = df.convert_dtypes(dtype_backend="numpy_nullable")

df

,group,value
0,group_a,0
1,group_a,2
2,group_b,1
3,group_b,3
4,group_b,5


In [22]:
df.groupby('group').sum()

,value
group,
group_a,2
group_b,9


从技术上讲，`pd.DataFrame.groupby` 将返回一个 `pd.core.groupby.DataFrameGroupBy` 对象，通过该对象公开 `pd.core.groupby.DataFrameGroupBy.sum` 来完成求和。

默认情况下，`pd.core.groupby.DataFrameGroupBy.sum` 被视为聚合，因此在 split‑apply‑combine 的应用阶段，每个组都会减少为一行。

我们可以使用 `pd.core.groupby.DataFrameGroupBy.agg` 方法，而不是直接调用 `pd.core.groupby.DataFrameGroupBy.sum`，并为其提供 `sum` 参数：

In [23]:
df.groupby('group').agg('sum')

,value
group,
group_a,2
group_b,9


与 `pd.core.groupby.DataFrameGroupBy.transform` 方法相比，`pd.core.groupby.DataFrameGroupBy.agg` 的明确性很有用，它将执行归约而不是转换：

In [24]:
df.groupby('group').transform('sum')

,value
0,2
1,2
2,9
3,9
4,9


`pd.core.groupby.DataFrameGroupBy.transform` 保证向调用者返回一个类似索引的对象，这使其非常适合执行类似组的 % 之类的计算：

In [25]:
df['value'].div(df.groupby('group')['value'].transform('sum'))

0         0.0
1         1.0
2    0.111111
3    0.333333
4    0.555556
Name: value, dtype: Float64

In [26]:
df[['value']].div(df.groupby('group')[['value']].transform('sum'))

,value
0,0.0
1,1.0
2,0.111111
3,0.333333
4,0.555556


当应用归约算法时，`pd.DataFrame.groupby` 将获取组的唯一值并使用它们形成新行 `pd.Index`（或在多个组的情况下为 `pd.MultiIndex`）。如果不希望分组标签创建新索引，而是将它们保留为列，则可以传递 `as_index=False`：

In [27]:
df.groupby('group', as_index=False).sum()

,group,value
0,group_a,2
1,group_b,9


还应注意，执行分组操作时，任何非分组列的名称都不会改变。例如，即使我们从包含名为 `value` 的列的 `pd.DataFrame` 开始：

In [28]:
df

,group,value
0,group_a,0
1,group_a,2
2,group_b,1
3,group_b,3
4,group_b,5


事实上，我们随后按组列进行分组并对值列求和不会在结果中改变其名称；它仍然只是 `value`：

In [29]:
df.groupby('group').sum()

,value
group,
group_a,2
group_b,9


如果将其他算法应用于您的组，这可能会造成混淆或歧义，例如 `min`：

In [30]:
df.groupby('group').min()

,value
group,
group_a,0
group_b,1


我们的列仍然被称为 `value`，即使在一个实例中，我们取值的总和，而在另一个实例中，我们取值的最小值。

有一种方法可以通过使用 `pd.NamedAgg` 类来控制这一点。当调用 `pd.core.groupby.DataFrameGroupBy.agg`，可以提供关键字参数，其中每个参数键表示所需的列名，参数值是 `pd.NamedAgg`，它表示聚合函数以及应用它的原始列。

In [31]:
df.groupby('group').agg(
    sum_of_value=pd.NamedAgg(column='value', aggfunc='mean'),
    min_of_value=pd.NamedAgg(column='value', aggfunc='min')
)

,sum_of_value,min_of_value
group,,
group_a,1.0,0
group_b,3.0,1


虽然这个配方主要侧重于求和，但 pandas 提供了许多其他内置的归约（agg）可以应用于 `pd.core.groupby.DataFrameGroupBy` 对象的算法，例如：

<table>
<tr>
    <td>any</td>
    <td>all</td>
    <td>sum</td>
    <td>prod</td>
</tr>
<tr>
    <td>idxmax</td>
    <td>idxmin</td>
    <td>min</td>
    <td>max</td>
</tr>
<tr>
    <td>mean</td>
    <td>median</td>
    <td>var</td>
    <td>std</td>
</tr>
<tr>
    <td>sem</td>
    <td>skew</td>
    <td>first</td>
    <td>last</td>
</tr>
</table>

同样，可以使用一些内置的 transform 函数：

<table>
<tr>
    <td>cumprod</td>
    <td>cumsum</td>
    <td>cummin</td>
</tr>
<tr>
    <td>cummax</td>
    <td>rank</td>
    <td></td>
</tr>
</table>

Functionally, there is no difference between calling these functions directly as methods of `pd.core.groupby.DataFrameGroupBy` versus providing them as an argument to `pd.core.groupby.DataFrameGroupBy.agg` or `pd.core.groupby.DataFrameGroupBy.transform`. You will get the same
performance and result by doing the following:

In [32]:
df.groupby('group').max()

,value
group,
group_a,2
group_b,5


In [33]:
df.groupby('group').agg('max')

,value
group,
group_a,2
group_b,5


In [34]:
df.groupby('group').transform('max')

,value
0,2
1,2
2,5
3,5
4,5
